# Evaluator-optimizer workflow with Pydantic AI

The evaluator-optimizer pattern uses a generator LLM and an evaluator LLM in a loop. The generator creates content, the evaluator checks it, and if it doesn't pass, feedback is sent back to the generator.

```mermaid
flowchart LR
    In([In]) --> Gen["Generator (LLM)"]
    Gen -- "Solution" --> Eval["Evaluator (LLM)"]
    Eval -- "Accepted" --> Out([Out])
    Eval -- "Rejected + Feedback" --> Gen
```

**Examples:**
- Content generation that must match certain guidelines (style, tone, language)
- Improving search results iteratively

In [ ]:
import nest_asyncio

nest_asyncio.apply()

## Setup

In [ ]:
import logfire
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pydantic_ai import Agent

load_dotenv()

logfire.configure()
logfire.instrument_pydantic_ai()

## Vanilla workflow

We have three agents:
- **Generator**: Creates an initial article
- **Fixer**: Improves the article based on feedback
- **Evaluator**: Checks if the article meets criteria (British English, appropriate for young audience, no em dashes)

In [ ]:
class Evaluation(BaseModel):
    explanation: str = Field(
        description="Explain why the text matches or not the evaluation criteria"
    )
    feedback: str = Field(
        description="Provide feedback to the writer to improve the text"
    )
    is_correct: bool = Field(
        description="Whether the text matches the evaluation criteria"
    )


generator = Agent(
    "openai:gpt-5-mini",
    system_prompt=(
        "You are an expert writer. Provided with a topic, "
        "you will generate an engaging article with less than 500 words."
    ),
)

fixer = Agent(
    "openai:gpt-5-mini",
    system_prompt=(
        "You are an expert writer. Provided with a text and feedback, "
        "you will improve the text."
    ),
)

evaluator = Agent(
    "openai:gpt-5-mini",
    system_prompt=(
        "You are an expert evaluator. Provided with a text, you will evaluate if it's written in "
        "British English and if it's appropriate for a young audience. The text must always use "
        "British spelling and grammar. Make sure the text doesn't include any em dashes."
    ),
    output_type=Evaluation,
)


def run_workflow(topic: str) -> str:
    text = generator.run_sync(f"Generate an article about '{topic}'").output

    for _ in range(3):
        evaluation = evaluator.run_sync(f"Evaluate the following text: {text}").output

        if evaluation.is_correct:
            return text

        text = fixer.run_sync(
            f"Fix the text: {text} with the following feedback: {evaluation.feedback}"
        ).output

    return text


output = run_workflow("Substance abuse of athletes")
print(output)

## Exercise

Transform the prompt chain workflow that generates recipes into an evaluator-optimizer workflow. It should make sure that the recipe is accurate, easy to follow, and that it has few ingredients.

In [ ]:
class Ingredients(BaseModel):
    ingredients: list[str] = Field(description="Ingredients needed for the recipe")


class RecipeDraft(BaseModel):
    ingredients: list[str]
    recipe_content: str


class RecipeEvaluation(BaseModel):
    explanation: str = Field(description="Why the recipe passed or failed")
    feedback: str = Field(description="How to improve the recipe")
    is_correct: bool = Field(description="Whether the recipe meets the criteria")
    score: int = Field(
        description="Quality score from 1 to 5",
        ge=1,
        le=5,
    )


ingredient_agent = Agent(
    "openai:gpt-5-mini",
    output_type=Ingredients,
    system_prompt=(
        "You are an expert chef. Given a recipe request, return the ingredient list."
    ),
)

recipe_generator = Agent(
    "openai:gpt-5-mini",
    system_prompt=(
        "You are an expert chef. Given a recipe request and ingredients, "
        "write a recipe that is accurate and easy to follow."
    ),
)

recipe_evaluator = Agent(
    "openai:gpt-5-mini",
    output_type=RecipeEvaluation,
    system_prompt=(
        "Evaluate recipes. A good recipe should be accurate, easy to follow, "
        "and use as few ingredients as possible, ideally 5 or fewer."
    ),
)

recipe_fixer = Agent(
    "openai:gpt-5-mini",
    output_type=RecipeDraft,
    system_prompt=(
        "Revise recipes using reviewer feedback. Improve clarity and accuracy, "
        "and remove unnecessary ingredients when possible."
    ),
)


def run_recipe_evaluator_optimizer(
    recipe_request: str,
    max_revisions: int = 2,
) -> dict:
    ingredients = ingredient_agent.run_sync(
        f"List the ingredients for {recipe_request}."
    ).output.ingredients

    recipe = recipe_generator.run_sync(
        f"Recipe request: {recipe_request}\nIngredients: {ingredients}"
    ).output

    evaluations = []
    for revision in range(max_revisions + 1):
        evaluation = recipe_evaluator.run_sync(
            f"Recipe request: {recipe_request}\n\n"
            f"Ingredients: {ingredients}\n\n"
            f"Recipe:\n{recipe}"
        ).output
        evaluations.append(evaluation)

        if evaluation.is_correct or revision == max_revisions:
            break

        revised = recipe_fixer.run_sync(
            f"Recipe request: {recipe_request}\n\n"
            f"Current ingredients: {ingredients}\n\n"
            f"Current recipe:\n{recipe}\n\n"
            f"Feedback:\n{evaluation.feedback}"
        ).output
        ingredients = revised.ingredients
        recipe = revised.recipe_content

    return {
        "ingredients": ingredients,
        "recipe": recipe,
        "evaluations": evaluations,
    }

In [ ]:
recipe_result = run_recipe_evaluator_optimizer("a recipe for a cake")

print("Final ingredients:", recipe_result["ingredients"])
print()
for i, evaluation in enumerate(recipe_result["evaluations"], start=1):
    print(f"Revision {i}: score={evaluation.score}, is_correct={evaluation.is_correct}")
    print("Feedback:", evaluation.feedback)
    print()

print(recipe_result["recipe"])